<a href="https://colab.research.google.com/github/TheFinix13/NLP-coursework/blob/main/notebooks/2.3_LoRA_Adapters_Mohamed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab: clone + pip install. Private repo: Colab secret GITHUB_TOKEN (classic PAT, repo) + Notebook access on.

import os
import sys
import shutil
import subprocess

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_DIR = os.environ.get("REPO_DIR", "/content/NLP-coursework")
REPO_SLUG = os.environ.get("GITHUB_REPO", "TheFinix13/NLP-coursework").strip().strip("/")
REPO_BRANCH = os.environ.get("REPO_BRANCH", "main").strip()
REPO_URL = os.environ.get("REPO_URL", "").strip()


def _token():
    t = os.environ.get("GITHUB_TOKEN", "").strip()
    if t:
        return t
    if not IN_COLAB:
        return ""
    try:
        from google.colab import userdata
        v = userdata.get("GITHUB_TOKEN")
        return "" if v is None else str(v).strip()
    except Exception:
        return ""


def _clone_url():
    if REPO_URL:
        return REPO_URL
    tok = _token()
    host = "github.com/" + REPO_SLUG + ".git"
    if tok:
        return "https://oauth2:" + tok + "@" + host
    return "https://" + host


def _is_repo_root(path):
    return bool(
        path
        and os.path.isdir(os.path.join(path, "src"))
        and os.path.isfile(os.path.join(path, "requirements.txt"))
    )


def _find_repo_under_content():
    base = "/content"
    if not os.path.isdir(base):
        return None
    for name in sorted(os.listdir(base)):
        p = os.path.join(base, name)
        if os.path.isdir(p) and _is_repo_root(p):
            return p
    return None


def _ensure_repo_colab():
    if _is_repo_root(REPO_DIR):
        return REPO_DIR

    zip_path = os.environ.get("REPO_ZIP", "/content/NLP-coursework.zip")
    if os.path.isfile(zip_path):
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "/content"], check=False)

    found = _find_repo_under_content()
    if found:
        return found

    if os.path.isdir(REPO_DIR) and not _is_repo_root(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    CLONE_URL = _clone_url()
    cmd = ["git", "clone", "--depth", "1"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [CLONE_URL, REPO_DIR]

    r = subprocess.run(cmd, capture_output=True, text=True)
    err = (r.stderr or r.stdout or "").strip()

    if r.returncode != 0 or not _is_repo_root(REPO_DIR):
        hint = ""
        if (
            "could not read Username" in err
            or "Authentication failed" in err
            or "403" in err
            or "Repository not found" in err
        ):
            hint = (
                "\nIf the repo is private: Colab → Secrets → GITHUB_TOKEN (repo scope) → enable Notebook access; "
                "Runtime → Restart, then re-run. Or upload NLP-coursework.zip to /content/.\n"
            )
        raise RuntimeError("Clone failed (need src/ + requirements.txt)." + hint + "\ngit said:\n" + err)
    return REPO_DIR


def _ensure_repo_local():
    cur = os.path.abspath(os.getcwd())
    for _ in range(10):
        if _is_repo_root(cur):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError("Run from repo root (contains src/ and requirements.txt).")


if IN_COLAB:
    root = _ensure_repo_colab()
    os.chdir(root)
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"],
        check=True,
    )
else:
    root = _ensure_repo_local()
    os.chdir(root)

os.environ["NLP_REPO_ROOT"] = os.path.abspath(root)
if root not in sys.path:
    sys.path.insert(0, root)
print("✅ Setup complete —", os.environ["NLP_REPO_ROOT"])



## Task 2.3 — LoRA adapters (Sarcasm)

This notebook trains LoRA adapters per variety (en-UK, en-AU, en-IN) on the **Sarcasm** task using a frozen base model in the **1–3B** range.

For a quick check, set `MODEL_KEY` to a smaller model (e.g. `opt-125m`). For coursework, use a 1–3B base (e.g. `qwen2.5-1.5b`). Set **`DEMO_MODE=0`** (env or below) for the full run, matching notebook 2.2.


In [ ]:
import os
import sys
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Repo root (Colab setup cell chdirs here; this also walks up if needed)
def _project_root():
    r = os.environ.get("NLP_REPO_ROOT")
    if r and os.path.isdir(os.path.join(r, "src")):
        return os.path.abspath(r)
    cur = os.path.abspath(os.getcwd())
    for _ in range(12):
        if os.path.isdir(os.path.join(cur, "src")) and os.path.isfile(os.path.join(cur, "requirements.txt")):
            return cur
        p = os.path.dirname(cur)
        if p == cur:
            break
        cur = p
    raise RuntimeError("Run the first setup cell or open the notebook from the repo root.")

PROJECT_ROOT = _project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.besstie_data_loader import get_variety_split
from src.training_utils import class_weights, new_weighted_class
from models.lora.lora_adapters import (
    LoRAConfig,
    load_model,
    apply_lora,
    tokenize_dataset,
    training_args,
)


In [ ]:
# Config (align with 2.2 RoBERTa: DEMO vs FULL)
DEMO_MODE = os.environ.get("DEMO_MODE", "1") == "1"

TASK = "Sarcasm"
VARIETIES = ["en-UK", "en-AU", "en-IN"]

MODEL_KEY = os.environ.get("MODEL_KEY", "qwen2.5-1.5b")

MAX_LENGTH = 128
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "4"))
LR = float(os.environ.get("LR", "2e-4"))

if DEMO_MODE:
    SEEDS = [42]
    EPOCHS = int(os.environ.get("EPOCHS", "1"))
    LIMIT_TRAIN = int(os.environ.get("LIMIT_TRAIN", "500"))
    LIMIT_VAL = int(os.environ.get("LIMIT_VAL", "200"))
    LIMIT_TEST = int(os.environ.get("LIMIT_TEST", "300"))
else:
    SEEDS = [42, 123]
    EPOCHS = int(os.environ.get("EPOCHS", "3"))
    LIMIT_TRAIN = int(os.environ.get("LIMIT_TRAIN", "0"))
    LIMIT_VAL = int(os.environ.get("LIMIT_VAL", "0"))
    LIMIT_TEST = int(os.environ.get("LIMIT_TEST", "0"))

print("Mode:", "DEMO" if DEMO_MODE else "FULL", "| MODEL_KEY=", MODEL_KEY, "| SEEDS=", SEEDS, "| EPOCHS=", EPOCHS)


In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")
print(ds)


In [ ]:
def maybe_limit(d, n: int, seed: int):
    if not n or n <= 0 or n >= len(d):
        return d
    return d.shuffle(seed=seed).select(range(n))


def train_one(variety: str, seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)

    train = get_variety_split(ds, variety, "train")
    val = get_variety_split(ds, variety, "validation")

    train = maybe_limit(train, LIMIT_TRAIN, seed)
    val = maybe_limit(val, LIMIT_VAL, seed)

    base_model, tokenizer = load_model(MODEL_KEY, num_labels=2)
    model = apply_lora(base_model, LoRAConfig())

    train_tok = tokenize_dataset(train, tokenizer, label_col=TASK, max_length=MAX_LENGTH)
    val_tok = tokenize_dataset(val, tokenizer, label_col=TASK, max_length=MAX_LENGTH)

    weights = class_weights(train, TASK)
    WeightedTrainer = new_weighted_class(weights)
    args = training_args(
        output_dir=f"./tmp/lora_{variety}_seed{seed}",
        variety=variety,
        seed=seed,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
    )
    args.report_to = "none"
    args.save_total_limit = 1

    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok)
    trainer.train()
    return trainer, tokenizer


In [ ]:
def evaluate(trainer, tokenizer, test_variety: str, seed: int):
    test = get_variety_split(ds, test_variety, "test")
    test = maybe_limit(test, LIMIT_TEST, seed)
    test_tok = tokenize_dataset(test, tokenizer, label_col=TASK, max_length=MAX_LENGTH)

    out = trainer.predict(test_tok)
    y_pred = np.argmax(out.predictions, axis=1)
    if out.label_ids is not None:
        y_true = np.asarray(out.label_ids).reshape(-1)
    else:
        y_true = np.asarray(test_tok["labels"])

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true,
        y_pred,
        target_names=["Not Sarcastic", "Sarcastic"],
        zero_division=0,
    )
    return {"macro_f1": float(macro_f1), "confusion_matrix": cm.tolist(), "report": report}


In [ ]:
# Train adapters per variety + evaluate on each test variety
results = {}
for variety in VARIETIES:
    results[variety] = {}
    for seed in SEEDS:
        trainer, tok = train_one(variety, seed)
        seed_res = {}
        for test_variety in VARIETIES:
            seed_res[test_variety] = evaluate(trainer, tok, test_variety, seed)
        results[variety][f"seed_{seed}"] = seed_res

results
